# 02-Generating Ground Truth Data

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
module_01_path = project_root / "01-agentic-rag-and-02-vector-search"

sys.path.append(str(module_01_path))

In [40]:
from ingest import load_faq_data, build_index
documents = load_faq_data()

In [4]:
#We'll generate questions only for the LLM Zoomcamp FAQ.

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

        

118

In [5]:
# I get a list with dictionaries, each dictionary representing a document. Each document has keys like 'course', 'question', and 'answer'.
documents_llm[3]

{'id': '04919992b3',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How should I start the course and follow the weekly workflow?',
 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'}

In [6]:
# We'll use these documents from now on so let's name them as documents 
documents = documents_llm

In [7]:
# Each document already has an id field:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## Generating questions with structured output

With structured output, we ask the LLM to return data in a specific format instead of free-form text. For example, instead of getting a paragraph that contains questions, we can ask for a Python object with a questions field.

In [8]:
# We want the output as a list of strings, so we define that structure with a Pydantic model

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['I just found out about the course — is it too late to join now?', 'Can I still start the course even if I missed the beginning?', 'If I join late, can I still get a certificate somehow?', 'What do I need to do to be eligible for the course certificate if I’m joining now?', 'Is it okay to enroll after the course has already started?']


In [15]:
print(result.questions)

['I just found out about the course — is it too late to join now?', 'Can I still start the course even if I missed the beginning?', 'If I join late, can I still get a certificate somehow?', 'What do I need to do to be eligible for the course certificate if I’m joining now?', 'Is it okay to enroll after the course has already started?']


## Reusable utilities

In [106]:
# Import the structured-output helper:
from evaluation_utils import llm_structured, calc_total_price

In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still join it now, or is it too late?', 'If I start late, am I still eligible for a certificate, and what do I need to do?', 'Do I have to finish everything on time to get the certificate, or is there a deadline for the project submission?', 'Can new students still enroll after the course has already started?', 'What’s the cutoff if I want the certificate after joining the course late?']


In [18]:
usage.input_tokens, usage.output_tokens

(207, 106)

In [19]:
from evaluation_utils import calc_price

In [20]:
cost = calc_price(usage)
cost

{'input_cost': 0.00015525, 'output_cost': 0.000477, 'total_cost': 0.00063225}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still join it now, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I start late, am I still eligible for a certificate, and what do I need to do?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to finish everything on time to get the certificate, or is there a deadline for the project submission?',
  'document': '74eb249bbf'},
 {'question': 'Can new students still enroll after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'What’s the cutoff if I want the certificate after joining the course late?',
  'document': '74eb249bbf'}]

# 03-Generating Ground Truth for All Documents

In [22]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [23]:
from tqdm.auto import tqdm
from evaluation_utils import llm_structured_retry

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [24]:
ground_truth

[{'question': 'I just found this course — is it still possible to join now?',
  'document': '74eb249bbf'},
 {'question': "Can I still enroll if I'm late to the course?",
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it started, can I still get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish and submit the project before submissions close to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I want the course certificate?',
  'document': '74eb249bbf'},
 {'question': 'I signed up for LLM Zoomcamp, but I never got any confirmation email. Should I be worried?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need a registration confirmation to start the course and submit homework?',
  'document': '977bf7786c'},
 {'question': 'Is there a list of approved students, or can anyone just begin the LLM Zoomcamp work?',
  'document': '977bf7786c'},
 {'question': 'What d

In [25]:
usages

[ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=88, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=295),
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=348),
 ResponseUsage(input_tokens=315, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=427),
 ResponseUsage(input_tokens=379, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=480),
 ResponseUsage(input_tokens=381, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=110, output_tokens

In [26]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [27]:
df_ground_truth.to_csv("ground_truth-new.csv", index=False)

## Search Evaluation

In [30]:
# Load the ground truth file from the previous notebook

df_ground_truth.head(2)

,question,document
0,I just found this course — is it still possibl...,74eb249bbf
1,Can I still enroll if I'm late to the course?,74eb249bbf


In [36]:
# is a pandas DataFrame method that converts a table into a list of dictionaries, where each row becomes one dictionary
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[0]

{'question': 'I just found this course — is it still possible to join now?',
 'document': '74eb249bbf'}

In [38]:
# remember we already have:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [ ]:
# search index = the organized memory of your documents that
# lets RAG find relevant context before asking the LLM
index = build_index(documents)

In [ ]:
# boost_dict means: give more importance to some fields when searching.
# return index.search(...) searches the index for documents relevant to the query and returns the top matching results.
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
   
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

## Collecting relevance data


In [46]:
# Run search for this question
doc_id = q["document"]
results = text_search(query=q["question"])
results

[{'id': 'c842475338',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Just found this course, can I still submit homeworks?',
  'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "Open/Submittable":\n\n- This is considered a "late homework submission," and the form is still editable.\n- Don’t forget to click the Update button to save any changes.\n\nPlease note, it\'s uncertain when the form will be closed as this process is currently manual.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you wan

In [47]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

c842475338 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
41aabbd7c5 == 74eb249bbf: False
2d8b16c2a0 == 74eb249bbf: False
3f1424af17 == 74eb249bbf: False


Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is the correct document for this question.

In [48]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[0, 1, 0, 0, 0]

In [49]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [50]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course — is it still possible to join now?


[0, 1, 0, 0, 0]

In [51]:

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [52]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [53]:
relevance_total_text

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0]]

Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

In [54]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

The total relevance function gets a search_function too

In [55]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [56]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0]]

In [57]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/25 [00:00<?, ?it/s]

In [59]:
relevance_total

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0]]

# Search Evaluation Metrics
In the previous lesson, we computed relevance lists for search results. We can turn those lists into metrics.

Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

## Hit Rate

In [60]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [61]:
# let's check it with an example 

example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [62]:
hit_rate(example)
# 0.933

0.9333333333333333

## Mean Reciprocal Rank (MRR)

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct document:

position 1: score is 1.0
position 2: score is 0.5
position 3: score is 0.333
not found: score is 0


In [63]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [64]:
mrr(example)
# 0.822

0.8222222222222222

## Putting it together


In [65]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [66]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/25 [00:00<?, ?it/s]

{'hit_rate': 0.56, 'mrr': 0.30466666666666664}

# Generating RAG Answers


In [67]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [75]:
first_doc = list(doc_idx.values())[0]
first_doc

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [76]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [77]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join now. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [78]:
assistant.total_cost()


0.000552

In [79]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [80]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course — is it still possible to join now?',
 'answer_llm': 'Yes — you can still join now. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

## Processing all questions


In [81]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [82]:
# Test it on one record:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — is it still possible to join now?',
 'answer_llm': 'Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [83]:
# Import the parallel processing helper from the same utility file:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
# Run RAG for all ground truth question
# Run RAG on many questions faster and in parallel, and show me the progress.

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/25 [00:00<?, ?it/s]

In [85]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [87]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("rag-answers-new.csv", index=False)

# LLM as a judge

In the previous lesson, we generated RAG answers for our ground truth questions. Now we need to decide whether these answers are good enough.

For offline evaluation, we have three things:

the original FAQ answer
the question generated from that answer
the answer generated by our RAG pipeline

## A->Q->A' evaluation

We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.

First, define the output format:

In [89]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [90]:
# First, write the judge instructions.

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [91]:
# Then define the prompt template. This is the data we pass to the judge for each answer.

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [92]:
# Take one record
rec = answers[0]

In [93]:
# Create the judge prompt
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [94]:
# Call the judge
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the original meaning: it says the course can still be joined, and that certificate eligibility depends on submitting the project while submissions are still open. This is semantically equivalent to the ground truth.', score='good')

In [96]:
# Check the cost
calc_price(usage)

{'input_cost': 0.000219, 'output_cost': 0.000261, 'total_cost': 0.00048}

In [97]:
# Now put the same logic into a function
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [99]:
# Test it on the same record
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: it says joining is still possible, and a certificate requires submitting the project while submissions are accepted. This is semantically equivalent.', score='good')

In [110]:
# Run the evaluation on all answers

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [102]:
# Use the same parallel processing helper

from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/25 [00:00<?, ?it/s]

In [103]:
# Split the results

evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [104]:
# Create a dataframe

df_eval = pd.DataFrame(evaluations)


In [107]:
# Calculate the total cost

calc_total_price(usages)

0.01631925

In [108]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 25/25 = 100.00%


In [109]:
# Look at the "bad" cases to understand what went wrong
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning


In [111]:
# Save the judged answers
df_eval.to_csv("rag-evaluations-new.csv", index=False)


In [112]:
df_eval

,question,document,score,reasoning
0,I just found this course — is it still possibl...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still enroll if I'm late to the course?,74eb249bbf,good,The AI answer preserves the key meaning of the...
2,"If I join the course after it started, can I s...",74eb249bbf,good,The AI answer preserves the core meaning of th...
3,Do I need to finish and submit the project bef...,74eb249bbf,good,The AI answer preserves the core point that a ...
4,What’s the deadline for project submission if ...,74eb249bbf,good,The ground truth says that to receive a certif...
5,"I signed up for LLM Zoomcamp, but I never got ...",977bf7786c,good,The AI answer preserves the key points from th...
6,Do I actually need a registration confirmation...,977bf7786c,good,The AI answer matches the ground truth: it say...
7,"Is there a list of approved students, or can a...",977bf7786c,good,The AI answer matches the ground truth: it say...
8,What does course registration do if it’s not r...,977bf7786c,good,The AI answer preserves the key points of the ...
9,"If I already registered, does that change anyt...",977bf7786c,good,The AI answer preserves the core meaning of th...
